# The dose responds to yesterday's outcome, and every adjustment set is wrong

This is the situation that breaks the standard playbook. A clinician raises the dose when the
patient got worse. A budget owner spends more where last week underperformed. So the
treatment at each period depends on the outcome at the last one, and now:

* **Do not adjust** for the lagged outcome, and it confounds the later doses.
* **Do adjust** for it, and you have conditioned on a mediator of the earlier doses, removing
  part of the effect you are trying to measure.

Both directions are wrong, from the same graph, and no single adjustment set exists that is
right. That is not a defect of the back-door criterion — it is why the g-formula exists.

`CausalGraph` carries a `feedback` flag whose only honest use is to say "the static summary
graph hides treatment–outcome feedback, so static adjustment is not enough". That flag is a
refusal. This is the answer to it: unroll the system over the periods you plan to analyse
and ask the ordinary graphical questions of the ordinary DAG that comes out.

In [ ]:
import numpy as np

from axiom.core import D, Param, dimensionless
from axiom.dynamics import Variable, parse_system
from axiom.identify import (
    CausalGraph, SequentialPlan, identify, sequential_backdoor_admissible, sequential_plan,
    unrolled_graph,
    unrolled_mixed_graph,
)
from axiom.viz import causal_graph

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, intervals

enable();  # every axiom result renders itself from here on

NONE = dimensionless()
system = parse_system(
    """
    outcome = beta * dose + rho * outcome[t-1] + kappa * frailty
    dose    = phi * outcome[t-1] + protocol
    """,
    variables=(
        Variable(name="outcome", dimension=D.outcome),
        Variable(name="dose", dimension=D.currency),
        Variable(name="protocol", dimension=D.currency, role="exogenous"),
        Variable(name="frailty", dimension=NONE, role="exogenous", observed=False),
    ),
    parameters=(
        Param(name="beta", dimension=D.outcome / D.currency),
        Param(name="rho", dimension=NONE),
        Param(name="kappa", dimension=D.outcome),
        Param(name="phi", dimension=D.currency / D.outcome),
    ),
    name="dose-responds-to-outcome",
)

## The unrolled graph

One node per variable per period; a variable declared `observed=False` is unmeasured at
every period. The result is an ordinary `CausalGraph`, so everything in `axiom.identify`
already works on it.

In [ ]:
graph = unrolled_graph(system, periods=3)
print(graph.nodes)
print(graph.to_text())
print("unmeasured:", graph.unmeasured)
print("topological order:", graph.topological_order())

In [ ]:
causal_graph(graph, height=460)

This is the picture the summary graph could not draw. Every arrow points forward in time, the
feedback is now a diagonal, and the reason no static adjustment set works is visible: the
outcome nodes in the middle are downstream of one dose and upstream of the next.

## Why one static adjustment set cannot work

`outcome.t1` is a **mediator** of `dose.t0` and a **confounder** of `dose.t2`. The static
back-door criterion demands adjusting for it and forbids adjusting for it, from the same
graph.

In [ ]:
print("descendant of dose.t0:", "outcome.t1" in graph.descendants("dose.t0"))
print("parent of dose.t2:", "outcome.t1" in graph.parents("dose.t2"))
static = identify(graph, "dose.t1", "outcome.t2")
print("static verdict for one stage:", static.verdict.status, static.route, static.adjustment_set)

## The sequential plan

`sequential_plan` runs the sequential back-door criterion stage by stage: at stage $k$ the
covariates must be measured non-descendants of $A_k \ldots A_n$, and $A_k$ must be
d-separated from the outcome given the history in the graph with all edges *out of*
$A_k \ldots A_n$ deleted.

Positivity is not a graphical property, so a plan that passes comes back **downgraded**,
not identified, carrying positivity as a named unverified assumption. A protocol that never
gives a low dose to a deteriorating patient has no data about what would have happened, and
no graph can see that.

In [ ]:
plan: SequentialPlan = sequential_plan(graph, ["dose.t0", "dose.t1", "dose.t2"], "outcome.t2")
print("status:", plan.verdict.status, "| licensed:", plan.licensed)
table(
    [
        [str(stage), str(extra or "()"), str(full or "()")]
        for stage, extra, full in zip(plan.stages, plan.adjustments, plan.conditioning_at)
    ],
    headers=("stage", "introduces", "conditions on"),
)
print("static adjustment fails:", plan.static_adjustment_fails)
print("assumptions:", [(a.name, a.state) for a in plan.verdict.assumptions])

In [ ]:
# Conditioning on a descendant of a later treatment is refused, with the reason.
ok, why = sequential_backdoor_admissible(
    graph, ["dose.t0", "dose.t1"], "outcome.t2", [("outcome.t1",), ()]
)
print(ok, "->", why)

## When no plan exists

Make the confounder persist *and* drive the dose, and no adjustment set closes the back
door at any stage. The verdict is blocked, and it names the stage and what would fix it.

In [ ]:
confounded = parse_system(
    """
    outcome = beta * dose + rho * outcome[t-1] + kappa * frailty
    dose    = phi * outcome[t-1] + protocol + omega * frailty
    frailty = persist * frailty[t-1]
    """,
    variables=(
        Variable(name="outcome", dimension=D.outcome),
        Variable(name="dose", dimension=D.currency),
        Variable(name="protocol", dimension=D.currency, role="exogenous"),
        Variable(name="frailty", dimension=NONE, observed=False),
    ),
    parameters=(
        Param(name="beta", dimension=D.outcome / D.currency),
        Param(name="rho", dimension=NONE),
        Param(name="kappa", dimension=D.outcome),
        Param(name="phi", dimension=D.currency / D.outcome),
        Param(name="omega", dimension=D.currency),
        Param(name="persist", dimension=NONE),
    ),
    name="unmeasured-time-varying-confounder",
)
blocked = sequential_plan(unrolled_graph(confounded, periods=3), ["dose.t0", "dose.t1", "dose.t2"], "outcome.t2")
print(blocked.verdict.status)
print(blocked.verdict.reason)

In [ ]:
# Measure the confounder and the same system becomes licensed.
measured = confounded.model_copy(update={
    "variables": tuple(
        v.model_copy(update={"observed": True}) if v.name == "frailty" else v
        for v in confounded.variables
    )
})
plan2 = sequential_plan(unrolled_graph(measured, periods=3), ["dose.t0", "dose.t1", "dose.t2"], "outcome.t2")
print(plan2.verdict.status, plan2.adjustments)

### That pair of verdicts, in the units of the effect

Two systems, one blocked and one licensed, are worth exactly as much as the difference they
make to the number. Below: data simulated from the confounded system — per-period effect
`beta = 1.5` — and the three regressions available to someone who has not asked the graph.

In [ ]:
truth = {"beta": 1.5, "rho": 0.5, "kappa": 2.0, "phi": -0.8, "omega": 1.2, "persist": 0.9}
n, periods = 6_000, 6
rng = np.random.default_rng(0)
frailty = rng.normal(size=n)
outcome_prev = rng.normal(size=n)
rows = []
for t in range(periods):
    frailty = truth["persist"] * frailty + rng.normal(0, 0.4, n)
    protocol = rng.normal(0, 1.0, n)
    dose = truth["phi"] * outcome_prev + protocol + truth["omega"] * frailty
    outcome = truth["beta"] * dose + truth["rho"] * outcome_prev + truth["kappa"] * frailty + rng.normal(0, 0.5, n)
    if t > 0:
        rows.append(np.column_stack([outcome, dose, outcome_prev, frailty]))
    outcome_prev = outcome
stacked = np.vstack(rows)
y, dose_col, lag_col, frailty_col = stacked.T


def slope(*regressors):
    design = np.column_stack([np.ones(len(y)), *regressors])
    coef, *_ = np.linalg.lstsq(design, y, rcond=None)
    resid = y - design @ coef
    var = (resid @ resid) / (len(y) - design.shape[1]) * np.linalg.inv(design.T @ design)[1, 1]
    return float(coef[1]), float(np.sqrt(var))


estimates = {
    "dose alone": slope(dose_col),
    "dose + lagged outcome  (the static fix)": slope(dose_col, lag_col),
    "…and the confounder, once measured": slope(dose_col, lag_col, frailty_col),
}
fig = intervals(
    [(label, est, est - 2 * se, est + 2 * se) for label, (est, se) in estimates.items()],
    ref=truth["beta"], ref_label="truth",
    highlight="dose + lagged outcome  (the static fix)",
    title="Blocked and licensed, in the units of the effect",
    subtitle="per-period effect of dose on outcome; 30,000 unit-periods of the confounded system",
    x_title="estimated beta",
)
caption(fig, "The middle row is the analysis a careful person writes after noticing the "
             "feedback — and it is the verdict that came back blocked. The bottom row is the "
             "same regression after measuring what the plan said had to be measured.")

## Simultaneity

A contemporaneous cycle is not a DAG and never will be. Its *reduced form* is, and that is
what `unrolled_graph` emits by default. Asking for the structural edges of a cyclic system
raises, naming the cycle — the honest outcome, since no DAG algorithm can answer a question
about it.

In [ ]:
market = parse_system(
    """
    quantity = a - b * price
    price    = d + e * quantity + cost
    """,
    variables=(
        Variable(name="quantity", dimension=D.outcome),
        Variable(name="price", dimension=D.currency),
        Variable(name="cost", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="a", dimension=D.outcome),
        Param(name="b", dimension=D.outcome / D.currency),
        Param(name="d", dimension=D.currency),
        Param(name="e", dimension=D.currency / D.outcome),
    ),
    name="market",
)
reduced = unrolled_graph(market, periods=2)
print("reduced:  ", reduced.to_text())
print("bidirected between the simultaneous variables:", reduced.bidirected)

structural = unrolled_mixed_graph(market, periods=2)
print()
print("as written:", structural.to_text())
print("cyclic?   ", not structural.is_acyclic, "| loops:", structural.cyclic_components)
print("and it still answers separation questions, by sigma-separation:")
print("   quantity indep price given the block's parents?",
      structural.sigma_separated("quantity.t0", "price.t0", ["cost.t0"]))

## What this bought you

The case where every adjustment set is wrong, turned into a stage-by-stage plan that says
what to condition on and when — or a refusal that names the stage and the variable that would
unblock it. Both come with positivity written down as an assumption rather than assumed
silently.

`nbs/design/05-structural.ipynb` designs an experiment against a system like this one, and
`nbs/identify/05-beyond-dags.ipynb` derives the same plan a second way, from SWIGs.